# Multi-agent systems: a hands-on implementation

**Track:** Agents · **Guide:** [`053-multi-agent.md`](../guides/concepts/053-multi-agent.md)

A multi-agent system assigns different scoped responsibilities to multiple cooperating agents or services.

> **Intuition:** A small project team separates research, drafting, and review instead of giving every person unrestricted authority.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Multi-agent systems]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** A research assistant gathers sources, a writer drafts a summary, and a verifier flags unsupported claims.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Give each role a narrow contract

A multi-agent workflow can use specialist roles, but each role should have explicit inputs and outputs.


In [ ]:
sources = [
    {"id": "policy-v3#p2", "text": "Unused items may be returned within 30 days."},
    {"id": "shipping-v2#p1", "text": "Standard delivery takes 3 to 5 business days."},
]

def researcher(question, source_documents):
    return [doc for doc in source_documents if "return" in doc["text"].lower()]

## 2. Draft and independently verify

Separate drafting from evidence checks to make unsupported claims easier to catch.


In [ ]:
def writer(question, evidence):
    if not evidence:
        return {"answer": "I don't know.", "citations": []}
    return {"answer": evidence[0]["text"], "citations": [evidence[0]["id"]]}

def verifier(draft, source_documents):
    source_ids = {doc["id"] for doc in source_documents}
    valid_citations = all(source in source_ids for source in draft["citations"])
    return {
        "approved": valid_citations and bool(draft["citations"]),
        "citation_check": valid_citations,
    }

evidence = researcher("What is the return period?", sources)
draft = writer("What is the return period?", evidence)
review = verifier(draft, sources)

## 3. Return a reviewed result

The orchestrator owns the final stop/approval decision. Additional agents add cost and coordination overhead, not guaranteed correctness.


In [ ]:
result = draft if review["approved"] else {
    "answer": "I could not verify an answer from the available sources.",
    "citations": [],
}
print({"result": result, "review": review})

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/053-multi-agent.md) and the
[learning roadmap](../guides/ROADMAP.md).
